# Reproduce competition submission — run_inference.py

**Before running, in the notebook settings (right sidebar):**
1. **Accelerator → GPU T4 x2**
2. **Internet → ON**  (needed for git clone + HuggingFace model download)
3. **Add Input → Competitions →** `cse-151-b-spring-2026-competition`

Then: run **Cell 1**, click **Run → Restart session**, then run **Cell 2**.

Cell 2 runs an 8-question SMOKE test first (~10 min incl. downloads). If it prints
`ROWS: 8` with two model loads and no OOM, set `SMOKE = False` and re-run Cell 2 to
produce the real 943-row `submission.csv` (~45-60 min) in `/kaggle/working/`.


In [ ]:
!pip install -q vllm==0.8.5 "transformers>=4.51,<5" "antlr4-python3-runtime==4.11.1" "protobuf<6.0"

# Kaggle T4: vLLM needs libcuda.so on the link path. Create a stub symlink.
import subprocess, os
cands = ["/usr/local/nvidia/lib64/libcuda.so.1", "/usr/lib/x86_64-linux-gnu/libcuda.so.1", "/usr/lib/libcuda.so.1"]
out = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout
cands += [l.split("=>")[1].strip() for l in out.splitlines() if "libcuda.so.1" in l]
libcuda = next((p for p in cands if os.path.exists(p) and "stubs" not in p), None) \
       or next((p for p in cands if os.path.exists(p)), None)
assert libcuda, "libcuda.so.1 not found"
stub_dir = "/kaggle/working/cuda_stubs"; os.makedirs(stub_dir, exist_ok=True)
stub = f"{stub_dir}/libcuda.so"
if os.path.lexists(stub): os.remove(stub)
os.symlink(libcuda, stub)
print("libcuda:", libcuda)
print(">>> Now click Run -> Restart session, then run Cell 2 <<<")

In [ ]:
import os, sys, subprocess

# T4 libcuda stub on the link path (inherited by the worker subprocesses)
for v in ("LIBRARY_PATH", "LD_LIBRARY_PATH"):
    os.environ[v] = "/kaggle/working/cuda_stubs:" + os.environ.get(v, "")

# Get the code (public repo)
REPO = "/kaggle/working/151B_SP26_Competition"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Trevis8688/151B_SP26_Competition.git", REPO], check=True)
sys.path.insert(0, REPO)
os.chdir(REPO)

PRIV = "/kaggle/input/competitions/cse-151-b-spring-2026-competition/private.jsonl"

# ---- SMOKE first (~10 min). Flip to False for the full ~45-60 min real run. ----
SMOKE = True

if SMOKE:
    import json
    rows = [json.loads(l) for l in open(PRIV)]
    sub = [r for r in rows if r.get("options")][:4] + [r for r in rows if not r.get("options")][:4]
    with open("/kaggle/working/smoke.jsonl", "w") as f:
        for r in sub:
            f.write(json.dumps(r) + "\n")
    os.environ["RUN_INFER_STAGE1_MAX_TOKENS"] = "48"   # force rescue path so BOTH models load
    os.environ["RUN_INFER_STAGE2_MAX_TOKENS"] = "96"
    priv, out = "/kaggle/working/smoke.jsonl", "/kaggle/working/smoke_submission.csv"
else:
    priv, out = PRIV, "/kaggle/working/submission.csv"

from run_inference import run_inference
run_inference(private_path=priv, output_csv=out, tensor_parallel_size=1)   # tp=1: T4-safe

import csv
with open(out, newline="") as f:
    n_rows = sum(1 for _ in csv.reader(f)) - 1  # -1 for header; csv handles quoted newlines
print("ROWS:", n_rows)